# Differential gene expression

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_dreampy
# python -m ipykernel install --user --name scrna_cartography_dreampy --display-name "dreampy"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc
import time

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# dream
import dreampy as dp

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi

In [2]:
# Paths
base_dir = str(here('data/annotate/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
diffg_dir = os.path.join(base_dir, 'dream_onevsother') 

anndata_dir = str(here('data/anndata/'))

mi.create_directories(os.path.join(base_dir, 'dream_onevsother'))

/work/islet_cartography_scrna/data/annotate/dream_onevsother Directory already exists!


In [3]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

## Differential gene expression

In [9]:
# Setup -----------------------------------------------------------------------------
anno_key   = "manual_annotation"
sample_key = "ic_id_platform_adjusted_sample"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"
target_celltypes = adata.obs[anno_key].unique()

#### Using all datasets

In [7]:
all_results = []

# Loop over all cell types -----------------------------------------------------------------------------
for cluster_id in target_celltypes:

    print(f"\n==============================")
    print(f"Running: {cluster_id} vs other")
    print(f"==============================")

    # Create binary column for current cluster vs others
    comp = f"{cluster_id}_vs_other"
    adata.obs[comp] = (
        adata
        .obs[anno_key]
        .apply(lambda x: cluster_id if x == cluster_id else 'other'))

    adata.obs['assay'] = 'my_assay'
    
    # Pseudobulk aggregation (by comparison group + sample)
    pb = dp.aggregate_pseudobulk(
        adata,
        layer='counts',
        groupby=['assay', sample_key, comp]
    )

    pb = dp.filter_samples(pb, min_cells=50, min_samples=3)
    pb = dp.compute_tmm_factors(pb, assay_col="assays")

    assays = dp.filter_by_expr(pb, assay_col="assays")

    # Define formula with donor as random effect
    formula = "~ {} + (1|{})".format(comp, donor_key)

    for name, assay_pb in assays.items():
        print(f"    Formula: {formula}") 
        comp_col = comp
        
        ref = "other"
        target_coef = f"{comp_col}_{cluster_id}"
        
        print(f"      Comparison: {cluster_id} vs other")
        # Enforce "other" as reference
        assay_pb.obs[comp_col] = assay_pb.obs[comp_col].astype("category")
        levels = list(assay_pb.obs[comp_col].cat.categories)
        assay_pb.obs[comp_col] = assay_pb.obs[comp_col].cat.reorder_categories(
            [ref] + [lvl for lvl in levels if lvl != ref],
            ordered=True)
    
        print(f"      Reference level: {ref}")

        assay_pb = dp.log2cpm(assay_pb)
        assay_pb = dp.estimate_weights(assay_pb, formula=formula, n_jobs = 60)
        fit = dp.fit_models(assay_pb, formula=formula, n_jobs= 60)
        fit_eb = dp.ebayes(fit)
        results = dp.get_results(fit_eb, assay_name=name)

        # Add metadata
        results["cell_type"] = cluster_id

        all_results.append(results)

# Combine all results -------------------------------------------------------------------------

marker_results = pd.concat(all_results)
marker_results.to_csv(os.path.join(diffg_dir, f"dreampy_one_vs_all.csv"), index=False)


Running: alpha vs other


/work/islet_cartography_scrna/scrna_cartography_dreampy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


filter_samples: 243 samples dropped (n_cells < 50)
  my_assay: 420 samples retained
filter_samples: 420/663 samples, 1 assays retained, 0 dropped
compute_tmm_factors: TMM normalization complete (1 assays, 420 samples)
filter_by_expr: filtering 1 assays
  my_assay: 17796/60656 genes retained (42860 filtered out)
    Formula: ~ alpha_vs_other + (1|ic_id_donor_overall)
      Comparison: alpha vs other
      Reference level: other
log2cpm: transformation complete (420 samples, 17796 genes)
Formula cleaned: 1 fixed effect(s), 1 random effect(s)
estimate_weights: Fitting 17796 genes with formula '~ alpha_vs_other + (1|ic_id_donor_overall)'...



Fitting genes: 100%|██████████| 17796/17796 [00:38<00:00, 460.56it/s]


estimate_weights: 17796/17796 genes converged with valid sigma
estimate_weights: Weights stored in .layers['weights']
estimate_weights: Voom info stored in .uns['voom_info']
Formula cleaned: 1 fixed effect(s), 1 random effect(s)


...

filter_samples: 213 samples dropped (n_cells < 50)
  my_assay: 258 samples retained
filter_samples: 258/471 samples, 1 assays retained, 0 dropped
compute_tmm_factors: TMM normalization complete (1 assays, 258 samples)
filter_by_expr: filtering 1 assays
  my_assay: 20733/60656 genes retained (39923 filtered out)
    Formula: ~ epsilon_vs_other + (1|ic_id_donor_overall)
      Comparison: epsilon vs other
      Reference level: other
log2cpm: transformation complete (258 samples, 20733 genes)
Formula cleaned: 1 fixed effect(s), 1 random effect(s)
estimate_weights: Fitting 20733 genes with formula '~ epsilon_vs_other + (1|ic_id_donor_overall)'...



Fitting genes: 100%|██████████| 20733/20733 [00:20<00:00, 1021.32it/s]


estimate_weights: 20733/20733 genes converged with valid sigma
estimate_weights: Weights stored in .layers['weights']
estimate_weights: Voom info stored in .uns['voom_info']
Formula cleaned: 1 fixed effect(s), 1 random effect(s)



Fitting genes: 100%|██████████| 20733/20733 [00:21<00:00, 975.84it/s] 


KeyError: 'celltype'

#### Per dataset

In [ ]:
# Loop over datasets and cell types -------------------------------------------------
meta_results = []

for cluster_id in target_celltypes:

    # define comparison
    comp_col = f"{cluster_id}_vs_other"

    for dataset in adata.obs[dataset_key].unique():
        # Preprocess -------------------------------------------------------------------------
        print(cluster_id, dataset)
        
        # Subset per dataset
        ad_sub = adata[
            adata.obs[dataset_key] == dataset
        ].copy()

        # Skip studies with less than 100 cells
        if ad_sub.n_obs < 100:
            continue

        # Create binary column for current cluster vs others
        comp = f"{cluster_id}_vs_other"
        ad_sub.obs[comp] = (
            ad_sub
            .obs[anno_key]
            .apply(lambda x: cluster_id if x == cluster_id else 'other'))
    
        ad_sub.obs['assay'] = 'my_assay'
        
        # Pseudobulk aggregation (by comparison group + sample)
        pb = dp.aggregate_pseudobulk(
            ad_sub,
            layer='counts',
            groupby=['assay', sample_key, comp]
        )

        try: 
            pb = dp.filter_samples(pb, min_cells=50, min_samples=3)
            pb = dp.compute_tmm_factors(pb, assay_col="assays")
        
            assays = dp.filter_by_expr(pb, assay_col="assays")
        
            # Define formula with donor as random effect
            formula = "~ {} + (1|{})".format(comp, donor_key)
        
            for name, assay_pb in assays.items():
                print(f"    Formula: {formula}") 
                comp_col = comp
                
                ref = "other"
                target_coef = f"{comp_col}_{cluster_id}"
                
                print(f"      Comparison: {cluster_id} vs other")
                
                # Enforce "other" as reference
                assay_pb.obs[comp_col] = assay_pb.obs[comp_col].astype("category")
                levels = list(assay_pb.obs[comp_col].cat.categories)
                assay_pb.obs[comp_col] = assay_pb.obs[comp_col].cat.reorder_categories(
                    [ref] + [lvl for lvl in levels if lvl != ref],
                    ordered=True)
            
                print(f"      Reference level: {ref}")
        
                assay_pb = dp.log2cpm(assay_pb)
                assay_pb = dp.estimate_weights(assay_pb, formula=formula, n_jobs = 60)
                fit = dp.fit_models(assay_pb, formula=formula, n_jobs= 60)
                fit_eb = dp.ebayes(fit)
                results = dp.get_results(fit_eb, assay_name=name)
                results["dataset"] = dataset
                results["cell_type"] = cluster_id

                meta_results.append(results)
        
        except Exception as e:
            print("Skipping:", dataset, e)
            
meta_df = pd.concat(meta_results)
meta_df.to_csv(os.path.join(diffg_dir, f"dreampy_one_vs_all_per_study.csv"), index=False)

alpha ic_15


/work/islet_cartography_scrna/scrna_cartography_dreampy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


  my_assay: 84 samples retained
filter_samples: 84/84 samples, 1 assays retained, 0 dropped
compute_tmm_factors: TMM normalization complete (1 assays, 84 samples)
filter_by_expr: filtering 1 assays
  my_assay: 19339/60656 genes retained (41317 filtered out)
    Formula: ~ alpha_vs_other + (1|ic_id_donor_overall)
      Comparison: alpha vs other
      Reference level: other
log2cpm: transformation complete (84 samples, 19339 genes)
Formula cleaned: 1 fixed effect(s), 1 random effect(s)
estimate_weights: Fitting 19339 genes with formula '~ alpha_vs_other + (1|ic_id_donor_overall)'...





Fitting genes:   0%|          | 0/19339 [00:00<?, ?it/s]


Fitting genes:   0%|          | 60/19339 [00:00<01:16, 251.25it/s]


Fitting genes:   1%|          | 120/19339 [00:03<09:19, 34.33it/s]


Fitting genes:   1%|          | 180/19339 [00:03<05:24, 59.03it/s]


Fitting genes:   2%|▏         | 300/19339 [00:03<02:29, 127.00it/s]


Fitting genes:   3%|▎         | 540/19339 [00:03<01:03, 297.95it/s]


Fitting genes:   5%|▌         | 1020/19339 [00:03<00:26, 696.72it/s]


Fitting genes:   8%|▊         | 1500/19339 [00:03<00:15, 1124.88it/s]


Fitting genes:  10%|█         | 1980/19339 [00:03<00:11, 1492.79it/s]


Fitting genes:  13%|█▎        | 2460/19339 [00:04<00:10, 1668.36it/s]


Fitting genes:  15%|█▌        | 2940/19339 [00:04<00:09, 1816.63it/s]


Fitting genes:  18%|█▊        | 3420/19339 [00:04<00:08, 1931.20it/s]


Fitting genes:  20%|██        | 3900/19339 [00:04<00:07, 2113.06it/s]


Fitting genes:  23%|██▎       | 4380/19339 [00:04<00:06, 2198.88it/s]


Fitting genes:  

estimate_weights: 19339/19339 genes converged with valid sigma
estimate_weights: Weights stored in .layers['weights']
estimate_weights: Voom info stored in .uns['voom_info']
Formula cleaned: 1 fixed effect(s), 1 random effect(s)


...




Fitting genes:   0%|          | 0/11479 [00:00<?, ?it/s]


Fitting genes:   2%|▏         | 240/11479 [00:00<00:05, 2040.02it/s]


Fitting genes:   5%|▌         | 600/11479 [00:00<00:05, 1997.05it/s]


Fitting genes:   7%|▋         | 840/11479 [00:00<00:05, 2076.23it/s]


Fitting genes:  11%|█▏        | 1320/11479 [00:00<00:04, 2480.27it/s]


Fitting genes:  16%|█▌        | 1800/11479 [00:00<00:04, 2278.01it/s]


Fitting genes:  20%|█▉        | 2280/11479 [00:01<00:04, 2124.38it/s]


Fitting genes:  24%|██▍       | 2760/11479 [00:01<00:04, 1970.71it/s]


Fitting genes:  28%|██▊       | 3240/11479 [00:01<00:04, 1883.18it/s]


Fitting genes:  32%|███▏      | 3720/11479 [00:01<00:04, 1890.63it/s]


Fitting genes:  37%|███▋      | 4200/11479 [00:02<00:06, 1168.69it/s]


Fitting genes:  41%|████      | 4680/11479 [00:03<00:05, 1184.66it/s]


Fitting genes:  45%|████▍     | 5160/11479 [00:03<00:04, 1322.36it/s]


Fitting genes:  49%|████▉     | 5640/11479 [00:03<00:04, 1415.75it/s]


Fitti

myeloid ic_9


/work/islet_cartography_scrna/scrna_cartography_dreampy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


filter_samples: 2 samples dropped (n_cells < 50)
  my_assay: 5 samples retained
filter_samples: 5/7 samples, 1 assays retained, 0 dropped
compute_tmm_factors: TMM normalization complete (1 assays, 5 samples)
filter_by_expr: filtering 1 assays
  my_assay: 12038/60656 genes retained (48618 filtered out)
    Formula: ~ myeloid_vs_other + (1|ic_id_donor_overall)
      Comparison: myeloid vs other
      Reference level: other
log2cpm: transformation complete (5 samples, 12038 genes)
Formula cleaned: 1 fixed effect(s), 1 random effect(s)
estimate_weights: Fitting 12038 genes with formula '~ myeloid_vs_other + (1|ic_id_donor_overall)'...





Fitting genes:   0%|          | 0/12038 [00:00<?, ?it/s]


Fitting genes:   2%|▏         | 240/12038 [00:00<00:07, 1591.77it/s]


Fitting genes:   5%|▍         | 600/12038 [00:00<00:05, 2006.21it/s]


Fitting genes:  11%|█         | 1320/12038 [00:00<00:03, 2773.40it/s]


Fitting genes:  15%|█▍        | 1800/12038 [00:00<00:03, 3035.28it/s]


Fitting genes:  19%|█▉        | 2280/12038 [00:00<00:03, 2711.86it/s]


Fitting genes:  23%|██▎       | 2760/12038 [00:01<00:03, 2805.87it/s]


Fitting genes:  27%|██▋       | 3240/12038 [00:01<00:03, 2838.75it/s]


Fitting genes:  31%|███       | 3720/12038 [00:01<00:02, 2875.61it/s]


Fitting genes:  35%|███▍      | 4200/12038 [00:01<00:02, 2885.46it/s]


Fitting genes:  39%|███▉      | 4680/12038 [00:01<00:02, 2915.13it/s]


Fitting genes:  43%|████▎     | 5160/12038 [00:01<00:02, 2957.84it/s]


Fitting genes:  47%|████▋     | 5640/12038 [00:01<00:02, 3062.42it/s]


Fitting genes:  51%|█████     | 6120/12038 [00:02<00:01, 3095.99it/s]


Fitt

In [ ]:
## Metafor
library(dplyr)
library(metafor)

df <- read.csv(
    "dreampy_meta_analysis_input.csv"
)

# Example: beta-cell markers only
beta_df <- df %>%
    filter(celltype == "beta")

# Run meta-analysis gene-by-gene
meta_results <- beta_df %>%
    group_by(id) %>%
    group_modify(~{

        dat <- .

        # require >=2 studies
        if(nrow(dat) < 2){
            return(NULL)
        }

        fit <- rma(
            yi  = dat$logfc,
            sei = dat$SE,
            method = "REML"
        )

        tibble(
            gene       = dat$id[1],
            meta_logFC = fit$b[1,1],
            se          = fit$se,
            pval        = fit$pval,
            I2          = fit$I2
        )
    })

meta_results <- meta_results %>%
    mutate(FDR = p.adjust(pval, method="fdr")) %>%
    arrange(FDR)

head(meta_results)

In [ ]:
# ----------------------------- SET UP -----------------------------------
print("Setting up variables")

anno_key         = "manual_annotation"
sample_key       = "ic_id_platform_adjusted_sample"
donor_key        = "ic_id_donor_overall"

# ----------------------------- PSEUDOBULK -----------------------------------
for cluster_id in adata.obs[anno_key].unique():

    # Create binary column for current cluster vs others
    comp = f"{cluster_id}_vs_other"
    adata.obs[comp] = (
        adata
        .obs[anno_key]
        .apply(lambda x: cluster_id if x == cluster_id else 'other'))

    adata.obs['assay'] = 'my_assay'
    
    # Pseudobulk aggregation (by comparison group + sample)
    pb = dp.aggregate_pseudobulk(
        adata,
        layer='counts',
        groupby=['assay', sample_key, comp]
    )

    pb = dp.filter_samples(pb, min_cells=50, min_samples=3)
    pb = dp.compute_tmm_factors(pb, assay_col="assays")

    assays = dp.filter_by_expr(pb, assay_col="assays")

    # Define formula with donor as random effect
    formula = "~ {} + (1|{})".format(comp, donor_key)

    for name, assay_pb in assays.items():
        print(f"    Formula: {formula}") 
        comp_col = comp
        
        ref = "other"
        target_coef = f"{comp_col}_{cluster_id}"
        
        print(f"      Comparison: {cluster_id} vs other")
        # Enforce "other" as reference
        assay_pb.obs[comp_col] = assay_pb.obs[comp_col].astype("category")
        levels = list(assay_pb.obs[comp_col].cat.categories)
        assay_pb.obs[comp_col] = assay_pb.obs[comp_col].cat.reorder_categories(
            [ref] + [lvl for lvl in levels if lvl != ref],
            ordered=True)
    
        print(f"      Reference level: {ref}")

        assay_pb = dp.log2cpm(assay_pb)
        assay_pb = dp.estimate_weights(assay_pb, formula=formula, n_jobs = 60)
        fit = dp.fit_models(assay_pb, formula=formula, n_jobs= 60)
        fit_eb = dp.ebayes(fit)
        results = dp.get_results(fit_eb, assay_name=name)

        results.to_csv(
        os.path.join(diffg_dir, f"dreampy_{cluster_id}_{target_coef}.csv"), 
        index=False)

In [ ]:
#(results
#    .loc[results["adj_p_val"] < 0.001]
#    .sort_values("logfc", ascending=False)
#    .head(20)
#)

In [ ]:
res = pd.read_csv(os.path.join(diffg_dir, "dreampy_ductal_mucin_ductal_mucin_vs_other_ductal_mucin.csv"))

In [ ]:
(res
    .loc[res["adj_p_val"] < 0.001]
    .sort_values(["logfc"], ascending=[False])
    .head(20)
)